# Stage 2 — Data Cleaning & Feature Derivation

**Adidas Global Catalogue 2026** · Path A + B · Governed by [`DOCS/STRUCTURE.md`](../DOCS/STRUCTURE.md) Stage 2.
Follows Stage 1 ([`01_ingestion.ipynb`](01_ingestion.ipynb)).

**Goal:** turn the immutable raw master into `data/processed/adidas_clean.parquet` with every
[`Adidas_Dataset_Analysis.md`](../DOCS/Adidas_Dataset_Analysis.md) **§3** data-quality defect fixed-or-flagged
and every **§7** derived column attached — then validate the result against a post-cleaning contract.

All logic is in [`src/cleaning.py`](../src/cleaning.py), [`src/features.py`](../src/features.py), and
[`src/validation.py`](../src/validation.py); this notebook orchestrates and narrates them. The three
external lookups live in [`data/external/`](../data/external/) ([README](../data/external/README.md)).

> **Iteration note (STRUCTURE):** Stage 2 ↔ Stage 3 is a loop. Anything EDA later surfaces returns here
> and is logged in `CHANGELOG.md`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "cleaning.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import ingestion, features as ft, cleaning, validation

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 170)
print("Project root:", ROOT)

Project root: C:\Users\HP ENVY\OneDrive\Documents\Personal Project\Dataset\Adidas Global Catalogue 2026


## 2.1 — External reference tables

Three hand-built lookups (not part of the raw data). Each is regenerated with an assertion that it
covers **every** raw value present in the master, so a new market value fails loudly rather than
silently dropping to null.

> **FX caveat:** `fx_rates.csv` holds early-2026 *reference approximations* pinned to 2026-03-09.
> Cross-market *relative* comparisons are robust; absolute USD claims need authoritative rates first.
> See [`data/external/README.md`](../data/external/README.md).

In [2]:
fx = ft.load_fx()
cat_map = ft.load_category_map()
gender_map = ft.load_gender_map()

print("FX rates (currency -> USD per unit):")
display(fx)
print(f"\ncategory_map: {len(cat_map)} raw -> {len(set(cat_map.values()))} canonical labels")
print("Canonical taxonomy:", sorted(set(cat_map.values())))
print(f"\ngender_map: {gender_map}")

FX rates (currency -> USD per unit):


,currency,usd_per_unit,as_of,source
0,USD,1.00000,2026-03-09,reference-approx
1,EUR,1.08000,2026-03-09,reference-approx
2,GBP,1.27000,2026-03-09,reference-approx
3,BRL,0.17000,2026-03-09,reference-approx
4,CNY,0.13800,2026-03-09,reference-approx
5,INR,0.01170,2026-03-09,reference-approx
6,JPY,0.00670,2026-03-09,reference-approx
7,KRW,0.00072,2026-03-09,reference-approx
8,MXN,0.04900,2026-03-09,reference-approx



category_map: 53 raw -> 17 canonical labels
Canonical taxonomy: ['Baseball', 'Basketball', 'Clothing', 'Cricket', 'Football', 'Golf', 'Gym & Training', 'Hiking', 'Lifestyle', 'Motorsport', 'Originals', 'Running', 'Shoes', 'Skateboarding', 'Tennis', 'Trail Running', 'Walking']

gender_map: {'K': 'K', 'M': 'M', 'U': 'U', 'W': 'W', '中性': 'U', '女子': 'W', '女童': 'K', '男子': 'M', '男童': 'K'}


## 2.2 — Run the cleaning pipeline

`cleaning.run()` loads the raw master (Stage 1), casts dtypes, derives the nine §7 columns, flags the
§3 defects, guards mapping coverage, and writes the parquet. The returned **cleaning log** is the
Stage 2 audit trail.

In [3]:
df, log = cleaning.run(save=True)
print(f"Clean frame: {df.shape[0]:,} rows x {df.shape[1]} cols")

Clean frame: 44,888 rows x 46 cols


In [4]:
from IPython.display import Markdown
Markdown(log.to_markdown())

## Stage 2 — Cleaning Log

- **Run (UTC):** 2026-07-12T13:37:37+00:00
- **Rows in → out:** 44,888 → 44,888

### Transformations
- Cast `snapshot_date`→datetime, `in_stock`→boolean, key categoricals→category, numerics enforced via to_numeric.
- §7 derived columns added: ['price_usd', 'category_canonical', 'gender_canonical', 'stock_coverage_pct', 'discount_pct_clean', 'is_cross_market', 'rating_count_clean', 'price_tier', 'snapshot_delta_flag'].
- §3#1 `rating_count_clean`: 8,338 `-99` sentinels → NaN.
- §3#6/#7 `discount_pct_clean` recomputed from prices: 711 negative source values and 19,012 rows with a sale price but source discount_pct==0 are now consistent (clipped ≥0).
- §3#3 `category_missing` flag: 517 rows (→ 'No Category' bucket). By market: {'CN': 517}.
- §3#9 grain preserved (size-level): 44,732 rows share a (snapshot, market, product_id) key — aggregate before treating product as unit.
- §3#4 kept-but-unusable columns (excluded from analysis): {'color_name': '90% null', 'badge_texts': '75% null'}.
- Mapping coverage guard passed (all raw category/gender values mapped).
- Saved → data\processed\adidas_clean.parquet (44,888 × 46).

**Read-out — every §3 defect is accounted for.** The log confirms the assessment's numbers exactly:
**8,338** `-99` rating sentinels neutralized; **711** negative + **19,012** inconsistent source discounts
recomputed into a clean `discount_pct_clean`; **517** CN rows routed to an explicit `'No Category'`
bucket (China stays in every cut); the size-level grain is preserved (never de-duped); and the two
90%/75%-null columns are kept for lineage but excluded from analysis.

## 2.3 — The nine §7 derived columns

Spot-check each derived column against its source so the transformations are transparent, not magic.

In [5]:
check_cols = ["country_code", "currency", "price_local", "price_usd",
              "category", "category_canonical", "gender_segment", "gender_canonical",
              "size_count", "available_size_count", "stock_coverage_pct",
              "sale_price_local", "discount_pct", "discount_pct_clean",
              "seen_market_count", "is_cross_market", "rating_count", "rating_count_clean",
              "price_tier", "snapshot_delta_flag"]
df[check_cols].head(10)

,country_code,currency,price_local,price_usd,category,category_canonical,gender_segment,gender_canonical,size_count,available_size_count,stock_coverage_pct,sale_price_local,discount_pct,discount_pct_clean,seen_market_count,is_cross_market,rating_count,rating_count_clean,price_tier,snapshot_delta_flag
0,BR,BRL,429.99,73.1,Lifestyle,Lifestyle,M,M,26,6.0,0.230769,549.99,0.0,0.0,1,False,-99.0,NaN,mid,False
1,BR,BRL,429.99,73.1,Lifestyle,Lifestyle,M,M,26,6.0,0.230769,549.99,0.0,0.0,1,False,-99.0,NaN,mid,False
2,BR,BRL,429.99,73.1,Lifestyle,Lifestyle,M,M,26,6.0,0.230769,549.99,0.0,0.0,1,False,-99.0,NaN,mid,False
3,BR,BRL,429.99,73.1,Lifestyle,Lifestyle,M,M,26,6.0,0.230769,549.99,0.0,0.0,1,False,-99.0,NaN,mid,False
4,BR,BRL,429.99,73.1,Lifestyle,Lifestyle,M,M,26,6.0,0.230769,549.99,0.0,0.0,1,False,-99.0,NaN,mid,False
5,BR,BRL,429.99,73.1,Lifestyle,Lifestyle,M,M,26,6.0,0.230769,549.99,0.0,0.0,1,False,-99.0,NaN,mid,False
6,BR,BRL,429.99,73.1,Lifestyle,Lifestyle,M,M,26,6.0,0.230769,549.99,0.0,0.0,1,False,-99.0,NaN,mid,False
7,BR,BRL,429.99,73.1,Lifestyle,Lifestyle,M,M,26,6.0,0.230769,549.99,0.0,0.0,1,False,-99.0,NaN,mid,False
8,BR,BRL,429.99,73.1,Lifestyle,Lifestyle,M,M,26,6.0,0.230769,549.99,0.0,0.0,1,False,-99.0,NaN,mid,False
9,BR,BRL,429.99,73.1,Lifestyle,Lifestyle,M,M,26,6.0,0.230769,549.99,0.0,0.0,1,False,-99.0,NaN,mid,False


In [6]:
# §3#1 sentinel fix: -99 gone from the clean column.
print("rating_count : min =", df['rating_count'].min(), "| has -99:", (df['rating_count'] == -99).any())
print("rating_count_clean: min =", df['rating_count_clean'].min(), "| has -99:", (df['rating_count_clean'] == -99).any())

# §3#6/#7 discount fix: no negatives remain; source vs clean disagreement quantified.
neg_src = (df['discount_pct'] < 0).sum()
print(f"\ndiscount_pct (source): {neg_src:,} negative rows | range "
      f"[{df['discount_pct'].min():.2f}, {df['discount_pct'].max():.2f}]")
print(f"discount_pct_clean   : {(df['discount_pct_clean'] < 0).sum():,} negative rows | range "
      f"[{df['discount_pct_clean'].min():.2f}, {df['discount_pct_clean'].max():.2f}]")

rating_count : min = -99.0 | has -99: True
rating_count_clean: min = 1.0 | has -99: False

discount_pct (source): 711 negative rows | range [-45.00, 60.00]
discount_pct_clean   : 0 negative rows | range [0.00, 0.48]


### Canonical category mix — the language problem, fixed

Before canonicalization, the same category appeared as `Football` / `Futebol` / `Fußball` / `サッカー` /
`足球`. After, it is one label — enabling the cross-market assortment cut in §5.

In [7]:
mix = (df.groupby('category_canonical', observed=True)
         .agg(rows=('country_code', 'size'), markets=('country_code', 'nunique'))
         .sort_values('rows', ascending=False))
mix

,rows,markets
category_canonical,,
Lifestyle,18583,10
Running,9973,10
Football,6463,9
Gym & Training,3518,9
Hiking,1240,6
Basketball,1227,9
Golf,1009,7
Shoes,683,5
Trail Running,661,5


In [8]:
# price_tier is a within-market tercile: budget/mid/premium should each be ~1/3 of a market.
pd.crosstab(df['country_code'], df['price_tier'], normalize='index').round(2)

price_tier,budget,mid,premium
country_code,,,
BR,0.42,0.34,0.24
CN,0.37,0.34,0.29
DE,0.34,0.38,0.27
FR,0.35,0.35,0.30
GB,0.35,0.36,0.29
IN,0.35,0.46,0.18
JP,0.35,0.38,0.27
KR,0.35,0.33,0.33
MX,0.35,0.33,0.32


## 2.4 — Validate the processed data (Stage 2 gate contract)

`src/validation.py` asserts the post-cleaning invariants (derived-column ranges, categorical domains,
grain integrity) and records expected missingness as warnings. A FAIL here blocks the gate.

In [9]:
result = validation.validate(df)
Markdown(result.to_markdown())

## Stage 2 — Validation Report

**Result: PASS** (0 failing checks)

- ✅ **stock_coverage_pct ∈ [0,1]** — all non-null values within [0, 1].
- ✅ **discount_pct_clean ∈ [0,1]** — all non-null values within [0, 1].
- ✅ **price_usd > 0** — all non-null values positive.
- ✅ **gender_canonical ∈ {M,W,U,K}** — all non-null values valid.
- ✅ **category_canonical in taxonomy** — all values within taxonomy.
- ✅ **grain uniqueness** — ['snapshot_date', 'country_code', 'product_id', 'size_label'] uniquely identifies every row.
- ⚠️ **rating coverage** — 61% null (expected — analysis uses valid subset).
- ⚠️ **rating_count_clean coverage** — 61% null (expected — analysis uses valid subset).
- ✅ **category mapping completeness** — every non-null raw category mapped.

In [10]:
assert result.ok, f"Validation failed: {result.failures}"
print("✅ Validation passed — no failing checks.")

✅ Validation passed — no failing checks.


## 2.5 — Persist validation evidence

In [11]:
report_path = ROOT / "reports" / "stage2_cleaning_report.md"
report_path.parent.mkdir(exist_ok=True)
report_path.write_text(log.to_markdown() + "\n\n" + result.to_markdown(), encoding="utf-8")
print("Wrote", report_path.relative_to(ROOT))
print("Processed data at:", (ROOT / 'data' / 'processed' / 'adidas_clean.parquet').relative_to(ROOT))

Wrote reports\stage2_cleaning_report.md
Processed data at: data\processed\adidas_clean.parquet


## Stage 2 — Gate Checklist

- [x] **Missing-value strategy documented per column** — `-99`→NaN (rating_count); category null → `'No Category'`;
      rating/rating_count kept (analysis uses valid subset); unusable color/badge kept for lineage only.
- [x] **Duplicates assessed** — size-level grain is intentional; `product_id` repeats flagged, never de-duped (§3 #9).
- [x] **All columns correctly typed** — dates→datetime, `in_stock`→boolean, key fields→category, numerics enforced.
- [x] **Business-rule validation passes** — `stock_coverage_pct`/`discount_pct_clean` ∈ [0,1], `price_usd`>0,
      `gender_canonical`∈{M,W,U,K}, `category_canonical` in taxonomy, grain unique.
- [x] **Cleaning log exists** — every transformation, with row counts and §-references (`reports/stage2_cleaning_report.md`).
- [x] **Cleaned data saved** — `data/processed/adidas_clean.parquet` (44,888 × 46).

### §3 ✔ and §7 ✔ — coverage confirmed
- **§3** all nine defects fixed-or-flagged (sentinel, localized category & gender, CN missing, negative/inconsistent discounts, unusable columns, rating coverage, grain).
- **§7** all nine derived columns present and validated.

**→ Gate passed. Proceed to Stage 3 — EDA (the five §5 descriptive cuts).**